In [53]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/nn-2026-fii-homework-1/sample_submission.csv
/kaggle/input/competitions/nn-2026-fii-homework-1/train.npz
/kaggle/input/competitions/nn-2026-fii-homework-1/test.npz


In [54]:
train=np.load('/kaggle/input/competitions/nn-2026-fii-homework-1/train.npz')
test=np.load('/kaggle/input/competitions/nn-2026-fii-homework-1/test.npz')


In [55]:
print(train)
print(test)

NpzFile '/kaggle/input/competitions/nn-2026-fii-homework-1/train.npz' with keys: data, labels
NpzFile '/kaggle/input/competitions/nn-2026-fii-homework-1/test.npz' with keys: data


In [56]:
train_data=train['data']
test_data=test['data']
train_labels=train['labels']

In [57]:
print(train_data)
print(train_data.shape)

[[ 70   0   0 ... 253   0  55]
 [  0   0   0 ... 238   0   0]
 [  0   0   0 ...   0   0   0]
 ...
 [  0   0   0 ... 108   0   0]
 [  0   0   0 ... 206   0   0]
 [  0   0   0 ...   0   0   0]]
(60000, 784)


In [58]:
print(test_data)
print(test_data.shape)

[[254   0   0 ...  22   0   0]
 [ 13   0   0 ...   0   0   0]
 [253   0   0 ... 253   0   0]
 ...
 [ 11   0   0 ...  10   0   0]
 [169   0   0 ... 254   0   0]
 [  0   0   5 ... 253   0   0]]
(15000, 784)


In [59]:
print(train_labels)
print(train_labels.shape)

[9 3 6 ... 9 5 7]
(60000,)


In [60]:
W=np.random.randn(784,10)*0.01
b=np.zeros(10)

X_train=train_data/255.0
X_test=test_data/255.0


In [61]:
m=train_labels.shape[0]
Y_train_one_hot=np.zeros((m,10))
Y_train_one_hot[np.arange(m),train_labels]=1
print(Y_train_one_hot.shape)

(60000, 10)


In [62]:
learning_rate=0.5
m=X_train.shape[0]
batch_size=64

for epochs in range(150):
    shuffle=np.random.permutation(m)
    X_shuffled=X_train[shuffle]
    Y_shuffled=Y_train_one_hot[shuffle]

    for i in range(0,m,batch_size):
        X_batch=X_shuffled[i:i+batch_size]
        Y_batch=Y_shuffled[i:i+batch_size]
        m_batch=X_batch.shape[0]
        
        Z=np.dot(X_batch,W)+b
        Z_max=np.max(Z, axis=1, keepdims=True)
        Z_shifted=Z-Z_max
        exp_Z=np.exp(Z_shifted)
        sum_exp_Z=np.sum(exp_Z, axis=1, keepdims=True)
        Y_pred_batch=exp_Z/sum_exp_Z
    
        err=Y_batch-Y_pred_batch
    
        W=W+learning_rate*(np.dot(X_batch.T,err)/m_batch)
        b=b+learning_rate*(np.sum(err,axis=0)/m_batch)

    if epochs%10==0:
        Z_full=np.dot(X_train,W)+b
        predictions=np.argmax(Z_full,axis=1)
        accuracy=np.mean(predictions==train_labels)*100
        print(f"Epoca {epochs}: Acuratețe pe antrenare = {accuracy:.2f}%")

    learning_rate=learning_rate*0.99

Epoca 0: Acuratețe pe antrenare = 91.28%
Epoca 10: Acuratețe pe antrenare = 92.25%
Epoca 20: Acuratețe pe antrenare = 91.41%
Epoca 30: Acuratețe pe antrenare = 92.43%
Epoca 40: Acuratețe pe antrenare = 93.41%
Epoca 50: Acuratețe pe antrenare = 93.66%
Epoca 60: Acuratețe pe antrenare = 93.60%
Epoca 70: Acuratețe pe antrenare = 93.69%
Epoca 80: Acuratețe pe antrenare = 93.56%
Epoca 90: Acuratețe pe antrenare = 93.70%
Epoca 100: Acuratețe pe antrenare = 93.85%
Epoca 110: Acuratețe pe antrenare = 93.80%
Epoca 120: Acuratețe pe antrenare = 93.82%
Epoca 130: Acuratețe pe antrenare = 93.86%
Epoca 140: Acuratețe pe antrenare = 93.85%


In [63]:
Z_test=np.dot(X_test, W)+b
test_predictions=np.argmax(Z_test,axis=1)

In [65]:
submission=pd.DataFrame({
    "ID":range(len(test_predictions)),
    "target":test_predictions
})
submission.to_csv('submission.csv',index=False)